# Discrete Scatter Transform  
This file implements one Layer of the Scatter-Transform in pytorch  
(batch, channel, time) => (batch, channel*2, time//2)  
Nyquist Theorem says to cut upper half the frequency band before downsampling by 2 => lowpass  
To preserve high frequency information, a complex highpass is applied  
The rapid oscillations of the phase stemming from imput signal deformation get removed by absolute value of the complex convolution  
This results in a high-frequency detector, which just tells us the strength of th amplitude  
It can be shown that this transformations preserves information  

The Scatter Transform changes the represention of a signal into a form which varies smoothly with deformation of the input  
The Features of a signal and its deformations appear now as simmilar  

The tolerace to deformation is 2^J, where J is the number of layers of scattering transform

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import numpy as np
from matplotlib import pyplot as plt

In [2]:
#compute all the wavelet coefficients

c0 = (1+np.sqrt(2))/6
c1 = (2*c0)/(6*c0-1)
c2 = (1+np.sqrt(3-np.power(2,1.5))-1/np.sqrt(2))/6
c3 = 0.5 - c1
c4 = 0.5 - c0 - c2

phi=np.array([c4,c3,c2,c1,2*c0,c1,c2,c3,c4])/2. #lowpass wavelet
n = np.arange(-4,4+1)
phase = np.exp(2*np.pi*1j/3*n)
psi = phase * phi #complex highpass wavelet

In [171]:
def stack_wavelets(c): #helper function to organise all the wavlets into on call to conv1d
    low     = phi [None,None,:] #lowpass wavlet
    high_re = np.real(psi)[None,None,:] #realpart of highpass
    high_im = np.imag(psi)[None,None,:] #imagpart of highpass
    stack   = np.concatenate([low,high_re,high_im]*c, axis=0)
    return torch.tensor(stack, dtype = torch.float32)

class Scatter(nn.Module):
    def __init__(self,c): #number of input channels; output has double the channels
        super(Scatter, self).__init__()
        self.c = c
        self.group_conv = nn.Conv1d(c, 3*c, 9, stride = 2, groups = c, bias = False, padding_mode = 'circular', padding = 8) # #valid convolution
        #fill the constant filters into the kernel
        self.group_conv.weight = torch.nn.Parameter(stack_wavelets(c), requires_grad = False) #weights must not be trainable !!!
        #print(self.group_conv.weight.shape)

    def forward(self, x):
        res = self.group_conv(x)
        #split result into the 3 components
        low     = res[:, 0::3]
        high_re = res[:, 1::3]
        high_im = res[:, 2::3]
        absolute = torch.sqrt(high_re**2 + high_im**2 +1e-32) #<-- this can have unstable gradients; solution: add an epsilon of 1e-8 into sqrt
        return torch.cat([low, absolute], axis = 1) #result doubles channels and halves time-resolution

# cut off below here  
## only tests

In [152]:
#plot wavelets
#plt.plot(phi)
plt.plot(np.real(psi))
plt.plot(np.imag(psi))
plt.plot(np.abs(psi))
plt.show()

In [172]:
#test the Layer on synthetic data
#batch, input_channel, time
bn = 1
cn = 1
tn = 256

#make a test signal
signal = torch.zeros(bn,cn,tn)
signal[:,:,64:64+32]=1
#signal[:,:,64] = 1
signal[:,:,65] = -1
#signal += 0.05*torch.randn(bn,cn,tn)

t = torch.linspace(0,20,steps = tn)[None,None,:]
#signal = np.exp(-0.5*t**2)*torch.sin(t*15)
#signal = np.where(t<5,t,0)
signal  = torch.cos(t**2)

#compute one layer of scatter transform
print(signal.shape)

sc = Scatter(signal.shape[1])
res = sc(signal)
print(res.shape)

sc2 = Scatter(res.shape[1])
res = sc2(res)
print(res.shape)


#plot the result


plt.plot(res[0,:].T)
    #plt.plot(signal[0,0])
plt.show()

torch.Size([1, 1, 256])
torch.Size([1, 2, 128])
torch.Size([1, 4, 64])


In [173]:
%matplotlib qt

import torch.optim as optim
from torch.autograd import Variable

N = 256

x = Variable(torch.randn(1,1,N), requires_grad=True)
optimizer = optim.SGD([x,], lr=1e-2)

sc1 = Scatter(1)
sc2 = Scatter(2)
sc3 = Scatter(4)
sc4 = Scatter(8)
#sc5 = Scatter(16)

for t in range(1000):
    optimizer.zero_grad()
    #print A.grad, b.grad
    y_pred = sc4(sc3(sc2(sc1(x))))
    #loss = -y_pred[0,9,:].sum() +(x**2).sum()
    loss = -y_pred[0,9,:].sum() +(x**2).sum()
    #loss = (y_pred**2).sum()-y_pred[0,15,:].sum()
    
    #x = x/ torch.sqrt(torch.sum(x**2))

    loss.backward()
    print(loss, end = '\r')
    optimizer.step()
    
plt.plot(x.detach()[0,0,:])
plt.show()